---
# Milestone 5: Model Evaluation & Analysis

1. **Full metric suite** — EM, BLEU, Parse Validity, Token-F1, ROUGE-L
2. **Pipeline ablation** — baseline vs RAG vs FSP vs CoT vs Full (head-to-head)
3. **FSP analysis** — example quality, diversity, token budget breakdown
4. **CoT analysis** — reasoning trace accuracy, step correctness, strip-and-compare
5. **Retrieval quality** — Recall@k, MRR, NDCG
6. **Per-complexity breakdown** — simple / medium / complex with all pipelines
7. **Error taxonomy** — 9-category failure classification, RAG+FSP+CoT vs baseline
8. **Clause-level accuracy** — per-SQL-clause Precision / Recall / F1
9. **Qualitative inspection** — best, near-miss, worst; what FSP/CoT fixed vs broke
10. **Attention visualisation** — cross-attention heatmap
11. **Limitations & improvement roadmap**


## 1  Setup — Load Best Model & Generate Predictions

In [ ]:
!pip install rouge-score -q
import matplotlib.pyplot as plt, matplotlib.patches as mpatches, seaborn as sns
from collections import Counter
from rouge_score import rouge_scorer as rouge_lib
plt.rcParams.update({'figure.dpi':120,'font.size':11})
sns.set_style('whitegrid')
print("M5 imports OK.")


In [ ]:
# ── Update to your best experiment ───────────────────────────────────────
BEST_EXP      = 'LoRA-BestConfig'   # update to winner
BEST_PIPELINE = 'full'
BEST_USE_LORA = True                 # True if best experiment used LoRA

CKPT = f"checkpoints/{BEST_EXP.replace(' ','_')}"
if not os.path.isdir(CKPT):
    dirs = [d for d in os.listdir('checkpoints') if os.path.isdir(f'checkpoints/{d}')]
    CKPT = f"checkpoints/{sorted(dirs)[0]}" if dirs else train_config['model_name']
    print(f"Fallback: {CKPT}")
    BEST_USE_LORA = False

if BEST_USE_LORA:
    from peft import PeftModel
    _base = T5ForConditionalGeneration.from_pretrained(train_config['model_name'])
    _base.resize_token_embeddings(len(tokenizer))
    adapter_dir = f"{CKPT}/lora_adapter"
    best_model = PeftModel.from_pretrained(_base, adapter_dir)
    print(f"Loaded LoRA model: {adapter_dir}")
    print(f"  Trainable params: {sum(p.numel() for p in best_model.parameters() if p.requires_grad):,}")
else:
    best_model = T5ForConditionalGeneration.from_pretrained(CKPT)
    best_model.resize_token_embeddings(len(tokenizer))
    print(f"Loaded full model: {CKPT}")

best_model = best_model.to(device)
best_model.eval()
gpu_snapshot("after loading best model")


In [ ]:
def generate_predictions(model, df: pd.DataFrame,
                          pipeline: str = 'full',
                          batch_size: int = 16, num_beams: int = 4) -> list:
    """
    Generate HiveQL for every row in df.
    pipeline controls what gets prepended (same logic as preprocessing).
    CoT trigger is added when pipeline in ('cot','full').
    strip_cot() is applied to every prediction before returning.
    """
    model.eval()
    preds = []
    for start in range(0, len(df), batch_size):
        batch = df.iloc[start:start+batch_size]
        inputs = []
        for q in batch['sqlite']:
            parts = [train_config['src_pfx'] + q]
            if pipeline in ('rag','fsp','cot','full'):
                chunks = retrieve_and_rerank(q)
                parts.append(train_config['ctx_pfx']+' '+build_context_string(chunks))
            if pipeline in ('fsp','full'):
                exs = retrieve_fsp_examples(q, n=train_config['fsp_n_shots'])
                parts.append(format_fsp_block(exs))
            if pipeline in ('cot','full'):
                parts.append(train_config['cot_trigger'])
            inputs.append(' '.join(parts))

        enc = tokenizer(inputs, max_length=train_config['max_ip_ln'],
                        padding=True, truncation=True, return_tensors='pt').to(device)
        with torch.no_grad():
            out = model.generate(
                input_ids=enc['input_ids'], attention_mask=enc['attention_mask'],
                num_beams=num_beams, max_new_tokens=train_config['max_new_tokens'],
                early_stopping=True)
        decoded = tokenizer.batch_decode(out, skip_special_tokens=True)
        preds.extend([strip_cot(p) for p in decoded])
    return preds

print("Generating predictions for all pipeline variants on test set...")
test_refs = test_df['hive'].tolist()

t0 = time.time()
preds_baseline = generate_predictions(best_model, test_df, pipeline='baseline')
preds_rag      = generate_predictions(best_model, test_df, pipeline='rag')
preds_fsp      = generate_predictions(best_model, test_df, pipeline='fsp')
preds_cot      = generate_predictions(best_model, test_df, pipeline='cot')
preds_full     = generate_predictions(best_model, test_df, pipeline='full')
print(f"All predictions done in {(time.time()-t0)/60:.1f} min")


---
## 2  Full Metric Suite

Five metrics evaluated for every pipeline variant.


In [ ]:
_rouge = rouge_lib.RougeScorer(['rougeL'], use_stemmer=False)

def token_f1(pred, ref):
    pt,rt = pred.lower().split(), ref.lower().split()
    if not pt or not rt: return 0.0
    n = sum((Counter(pt)&Counter(rt)).values())
    if n==0: return 0.0
    p,r = n/len(pt), n/len(rt)
    return 2*p*r/(p+r)

def compute_full_metrics(preds, refs, label=''):
    cp   = [strip_cot(p) for p in preds]
    np_  = [normalise_sql(p) for p in cp]
    nr   = [normalise_sql(r) for r in refs]
    em   = sum(p==r for p,r in zip(np_,nr)) / max(len(np_),1)
    bleu = bleu_metric.compute(predictions=np_,references=[[r] for r in nr])['score']
    pv   = sum(is_valid_hive(p) for p in cp) / max(len(cp),1)
    tf1  = np.mean([token_f1(p,r) for p,r in zip(np_,nr)])
    rl   = np.mean([_rouge.score(r,p)['rougeL'].fmeasure for p,r in zip(np_,nr)])
    return {'label':label, 'EM%':round(em*100,2), 'BLEU':round(bleu,2),
            'PV%':round(pv*100,2), 'TF1%':round(tf1*100,2), 'RL%':round(rl*100,2),
            '_tf1_list':[token_f1(p,r) for p,r in zip(np_,nr)]}

pipeline_metrics = {
    'baseline' : compute_full_metrics(preds_baseline, test_refs, 'Baseline'),
    'rag'      : compute_full_metrics(preds_rag,      test_refs, 'RAG'),
    'fsp'      : compute_full_metrics(preds_fsp,      test_refs, 'RAG+FSP'),
    'cot'      : compute_full_metrics(preds_cot,      test_refs, 'RAG+CoT'),
    'full'     : compute_full_metrics(preds_full,     test_refs, 'RAG+FSP+CoT'),
}

summary = pd.DataFrame([{k:v for k,v in m.items() if not k.startswith('_')}
                         for m in pipeline_metrics.values()])
print("\n"+"="*65+"\n  PIPELINE COMPARISON — TEST SET\n"+"="*65)
print(summary.to_string(index=False))
print("="*65)


In [ ]:
# ── Grouped bar chart: all pipelines × all metrics ───────────────────────
metrics_plot = ['EM%','BLEU','PV%','TF1%','RL%']
labels_plot  = [m['label'] for m in pipeline_metrics.values()]
colors       = ['#4C72B0','#DD8452','#55A868','#C44E52','#9B59B6']

x = np.arange(len(metrics_plot)); w = 0.15
fig, ax = plt.subplots(figsize=(13,5))
for i,(pipe,m) in enumerate(pipeline_metrics.items()):
    vals = [m[k] for k in metrics_plot]
    bars = ax.bar(x + i*w, vals, w, label=m['label'], color=colors[i], alpha=0.85)
    ax.bar_label(bars, fmt='%.1f', fontsize=7, padding=2, rotation=0)

ax.set_xticks(x + w*2); ax.set_xticklabels(metrics_plot)
ax.set_ylabel('Score'); ax.set_ylim(0,115)
ax.set_title('Pipeline Comparison — All Metrics (Test Set)', fontweight='bold')
ax.legend(bbox_to_anchor=(1.01,1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig('m5_pipeline_comparison.png', bbox_inches='tight')
plt.show()
print("Saved → m5_pipeline_comparison.png")


---
## 3  Few-Shot Prompting Analysis

### 3a  Token budget breakdown
### 3b  Example diversity (how often retrieved examples share clauses with query)
### 3c  FSP contribution — per-complexity lift over RAG-only baseline


In [ ]:
# 3a. Token budget breakdown on test samples
sample_n = min(100, len(test_df))
budget_rows = []
for i in range(sample_n):
    q   = test_df['sqlite'].iloc[i]
    chunks = retrieve_and_rerank(q)
    exs    = retrieve_fsp_examples(q, n=2)
    base_text = train_config['src_pfx'] + q
    rag_text  = base_text + ' ' + train_config['ctx_pfx'] + ' ' + build_context_string(chunks)
    fsp_text  = rag_text  + ' ' + format_fsp_block(exs)
    full_text = fsp_text  + ' ' + train_config['cot_trigger']
    budget_rows.append({
        'base_toks' : tokenizer(base_text,  return_length=True)['length'][0],
        'rag_toks'  : tokenizer(rag_text,   return_length=True)['length'][0],
        'fsp_toks'  : tokenizer(fsp_text,   return_length=True)['length'][0],
        'full_toks' : tokenizer(full_text,  return_length=True)['length'][0],
    })
budget_df = pd.DataFrame(budget_rows)
print("Token budget (mean ± std across first 100 test samples):")
for col in ['base_toks','rag_toks','fsp_toks','full_toks']:
    print(f"  {col:12s}: {budget_df[col].mean():.0f} ± {budget_df[col].std():.0f}  "
          f"  max={budget_df[col].max()}  pct>512={( budget_df[col]>512).mean()*100:.1f}%")


In [ ]:
# 3b. Example diversity — clause overlap between query and retrieved examples
SQL_CLAUSES_SET = {'select','from','where','group by','having',
                   'order by','limit','join','union'}
def get_clauses(sql):
    s = sql.lower()
    return {c for c in SQL_CLAUSES_SET
            if re.search(r'\b'+c.replace(' ',r'\s+')+r'\b', s)}

overlap_scores = []
for q in test_df['sqlite'].head(200):
    q_clauses = get_clauses(q)
    exs = retrieve_fsp_examples(q, n=2)
    for ex in exs:
        ex_clauses = get_clauses(ex['sqlite'])
        if q_clauses:
            overlap_scores.append(len(q_clauses & ex_clauses) / len(q_clauses))

print(f"FSP example clause overlap (mean): {np.mean(overlap_scores):.3f}")
print(f"  (1.0 = examples share all clauses with query)")

fig, ax = plt.subplots(figsize=(7,3))
ax.hist(overlap_scores, bins=10, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(np.mean(overlap_scores), color='red', lw=2, linestyle='--',
           label=f"Mean={np.mean(overlap_scores):.2f}")
ax.set_xlabel('Clause overlap ratio'); ax.set_ylabel('Count')
ax.set_title('FSP Example Clause Overlap vs Query', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('m5_fsp_clause_overlap.png', bbox_inches='tight')
plt.show()
print("Saved → m5_fsp_clause_overlap.png")


In [ ]:
# 3c. FSP lift over RAG — per complexity
if 'complexity' in test_df.columns:
    fsp_lift = []
    for cplx, grp in test_df.groupby('complexity'):
        idx = grp.index.tolist()
        rag_em  = np.mean([normalise_sql(preds_rag[i])==normalise_sql(test_refs[i])  for i in idx])
        fsp_em  = np.mean([normalise_sql(preds_fsp[i])==normalise_sql(test_refs[i])  for i in idx])
        full_em = np.mean([normalise_sql(preds_full[i])==normalise_sql(test_refs[i]) for i in idx])
        fsp_lift.append({'Complexity':cplx, 'Count':len(idx),
                         'RAG EM%':round(rag_em*100,2),
                         'FSP EM%':round(fsp_em*100,2),
                         'Full EM%':round(full_em*100,2),
                         'FSP delta':round((fsp_em-rag_em)*100,2),
                         'Full delta':round((full_em-rag_em)*100,2)})
    lift_df = pd.DataFrame(fsp_lift)
    print("\nFSP lift over RAG by complexity:")
    print(lift_df.to_string(index=False))


---
## 4  Chain-of-Thought Analysis

We evaluate whether the CoT traces are: (a) correctly stripped before metrics,
(b) structurally coherent, and (c) correlated with translation accuracy.


In [ ]:
# 4a. Strip verification — ensure strip_cot removes traces correctly
cot_raw_sample = generate_predictions(best_model, test_df.head(10),
                                       pipeline='cot', num_beams=4)
# Re-generate WITHOUT strip to inspect raw output
def generate_raw(model, df, pipeline='cot', batch_size=8, num_beams=4):
    """Same as generate_predictions but returns raw text (no strip_cot)."""
    model.eval()
    preds = []
    for start in range(0, len(df), batch_size):
        batch = df.iloc[start:start+batch_size]
        inputs = []
        for q in batch['sqlite']:
            parts = [train_config['src_pfx'] + q]
            if pipeline in ('rag','fsp','cot','full'):
                chunks = retrieve_and_rerank(q)
                parts.append(train_config['ctx_pfx']+' '+build_context_string(chunks))
            if pipeline in ('cot','full'):
                parts.append(train_config['cot_trigger'])
            inputs.append(' '.join(parts))
        enc = tokenizer(inputs,max_length=512,padding=True,
                        truncation=True,return_tensors='pt').to(device)
        with torch.no_grad():
            out = model.generate(input_ids=enc['input_ids'],
                                  attention_mask=enc['attention_mask'],
                                  num_beams=num_beams,max_new_tokens=256)
        preds.extend(tokenizer.batch_decode(out, skip_special_tokens=True))
    return preds

raw_outputs = generate_raw(best_model, test_df.head(10), pipeline='cot')

cot_sep = train_config['cot_sep'].strip()
has_trace  = sum(1 for p in raw_outputs if cot_sep in p)
clean_same = sum(1 for p,c in zip(raw_outputs, cot_raw_sample)
                 if strip_cot(p)==c)

print(f"Raw outputs that contain '{cot_sep}': {has_trace}/10")
print(f"strip_cot correctly extracts HiveQL:  {clean_same}/10")
print()
print("Sample raw CoT outputs:")
for i, (raw, clean) in enumerate(zip(raw_outputs[:3], cot_raw_sample[:3])):
    print(f"  [{i+1}] RAW  : {raw[:150]}")
    print(f"       CLEAN: {clean[:100]}")
    print()


In [ ]:
# 4b. CoT step correctness — does each step mention a real dialect difference?
KNOWN_DIFFERENCES = [
    'BIGINT','DOUBLE','STRING','COLLECT_LIST','COLLECT_SET',
    'LATERAL VIEW','EXPLODE','DISTRIBUTE BY','CLUSTER BY',
    'from_unixtime','date_format','RLIKE','ORC','Parquet'
]

def count_valid_steps(trace: str) -> int:
    """Count how many Step lines reference a known HiveQL difference."""
    steps = re.findall(r'Step \d+:[^.]+\.', trace)
    return sum(1 for s in steps
               if any(kw.lower() in s.lower() for kw in KNOWN_DIFFERENCES))

raw_full = generate_raw(best_model, test_df.head(50), pipeline='full')
traces   = [p.split(cot_sep)[0] if cot_sep in p else '' for p in raw_full]

valid_steps = [count_valid_steps(t) for t in traces if t]
total_steps = [len(re.findall(r'Step \d+:',t)) for t in traces if t]

print(f"CoT trace analysis ({len(valid_steps)} samples with traces):")
print(f"  Avg steps per trace         : {np.mean(total_steps):.1f}")
print(f"  Avg valid dialect references: {np.mean(valid_steps):.1f}")
if total_steps:
    pct = sum(v/t if t>0 else 0 for v,t in zip(valid_steps,total_steps)) / len(total_steps)
    print(f"  Step validity rate          : {pct*100:.1f}%")


In [ ]:
# 4c. Does having a CoT trace correlate with higher translation accuracy?
cot_em_with    = []
cot_em_without = []

raw_all = generate_raw(best_model, test_df.head(200), pipeline='full')
for i, (raw, ref) in enumerate(zip(raw_all, test_refs[:200])):
    has_reasoning = cot_sep in raw
    em = normalise_sql(strip_cot(raw)) == normalise_sql(ref)
    if has_reasoning:
        cot_em_with.append(em)
    else:
        cot_em_without.append(em)

print("EM accuracy correlation with CoT trace presence:")
if cot_em_with:
    print(f"  Rows WITH reasoning trace  : {np.mean(cot_em_with)*100:.1f}%  (n={len(cot_em_with)})")
if cot_em_without:
    print(f"  Rows WITHOUT trace         : {np.mean(cot_em_without)*100:.1f}%  (n={len(cot_em_without)})")


---
## 5  Retrieval Quality Metrics

In [ ]:
def heuristic_relevance(query, ref_hive, chunk_text):
    stopw = {'select','from','where','and','or','not','in','is','as','on',
             'by','the','a','an','of','for','to','with','null'}
    def kw(t):
        return {w.lower() for w in re.findall(r'[a-zA-Z_][a-zA-Z0-9_]*',t)
                if len(w)>3 and w.lower() not in stopw}
    return len(kw(ref_hive) & kw(chunk_text)) >= 2

def compute_retrieval_metrics(queries, refs, k_values=(1,3,5)):
    recall = {k:[] for k in k_values}; mrr=[]; ndcg=[]
    for q,ref in zip(queries,refs):
        cands = retrieve(q, k=max(k_values))
        rel   = [heuristic_relevance(q,ref,c['text']) for c in cands]
        for k in k_values: recall[k].append(float(any(rel[:k])))
        rr=0.0
        for rank,r in enumerate(rel,1):
            if r: rr=1/rank; break
        mrr.append(rr)
        dcg  = sum(r/np.log2(i+2) for i,r in enumerate(rel[:5]))
        idcg = sum(1/np.log2(i+2) for i in range(min(sum(rel[:5]),5)))
        ndcg.append(dcg/idcg if idcg>0 else 0)
    res = {f'Recall@{k}':round(np.mean(v)*100,2) for k,v in recall.items()}
    res['MRR']    = round(np.mean(mrr)*100,2)
    res['NDCG@5'] = round(np.mean(ndcg)*100,2)
    return res

N = min(200, len(test_df))
print(f"Computing retrieval metrics on {N} samples...")
ret_metrics = compute_retrieval_metrics(test_df['sqlite'][:N], test_refs[:N])
print("\n"+"="*45+"\n  RETRIEVAL QUALITY\n"+"="*45)
for k,v in ret_metrics.items(): print(f"  {k:12s}: {v}%")
print("="*45)

fig,ax = plt.subplots(figsize=(7,4))
bars = ax.bar(ret_metrics.keys(), ret_metrics.values(),
              color='steelblue', edgecolor='white', alpha=0.85)
ax.bar_label(bars, fmt='%.1f%%', fontsize=10, padding=3)
ax.set_ylim(0,115); ax.set_ylabel('Score (%)')
ax.set_title('Retrieval Pipeline Quality (Test Sample)', fontweight='bold')
plt.tight_layout()
plt.savefig('m5_retrieval_quality.png', bbox_inches='tight')
plt.show()


---
## 6  Per-Complexity Breakdown — All Pipelines

In [ ]:
test_eval = test_df.reset_index(drop=True).copy()
for pipe, preds in [('baseline',preds_baseline),('rag',preds_rag),
                     ('fsp',preds_fsp),('cot',preds_cot),('full',preds_full)]:
    test_eval[f'em_{pipe}']  = [normalise_sql(strip_cot(p))==normalise_sql(r)
                                  for p,r in zip(preds, test_refs)]
    test_eval[f'pv_{pipe}']  = [is_valid_hive(strip_cot(p)) for p in preds]
    test_eval[f'tf1_{pipe}'] = [token_f1(normalise_sql(strip_cot(p)),normalise_sql(r))
                                  for p,r in zip(preds, test_refs)]

if 'complexity' in test_eval.columns:
    rows=[]
    for cplx,grp in test_eval.groupby('complexity'):
        row = {'Complexity':cplx,'N':len(grp)}
        for pipe in ['baseline','rag','fsp','cot','full']:
            row[f'EM-{pipe}%'] = round(grp[f'em_{pipe}'].mean()*100,2)
        rows.append(row)
    cplx_df = pd.DataFrame(rows)
    print("Exact Match by complexity and pipeline:")
    print(cplx_df.to_string(index=False))

    fig, axes = plt.subplots(1,3,figsize=(16,5))
    pipes  = ['baseline','rag','fsp','cot','full']
    colors = ['#4C72B0','#DD8452','#55A868','#C44E52','#9B59B6']
    x      = np.arange(len(cplx_df)); w = 0.15
    for ax_i,(ax,metric,title) in enumerate(zip(axes,[
        ('em','Exact Match %'),('tf1','Token F1 %'),('pv','Parse Validity %')
    ])):
        metric_key, title = metric
        for j,(pipe,color) in enumerate(zip(pipes,colors)):
            vals = [grp[f'{metric_key}_{pipe}'].mean()*100
                    for _,grp in test_eval.groupby('complexity')]
            b = ax.bar(x+j*w, vals, w, label=pipe, color=color, alpha=0.85)
            ax.bar_label(b, fmt='%.0f', fontsize=7, padding=1)
        ax.set_xticks(x+w*2); ax.set_xticklabels(cplx_df['Complexity'])
        ax.set_title(title, fontweight='bold'); ax.set_ylim(0,115)
        if ax_i==0: ax.legend(fontsize=8)
    plt.suptitle('Per-Complexity Performance — All Pipelines',fontsize=12,fontweight='bold')
    plt.tight_layout()
    plt.savefig('m5_complexity_all_pipelines.png', bbox_inches='tight')
    plt.show()
    print("Saved → m5_complexity_all_pipelines.png")


---
## 7  Error Taxonomy — Full Pipeline vs Baseline

In [ ]:
SQL_CLAUSES_E = ['select','from','where','group by','having','order by',
                  'limit','join','left join','inner join','union','with']

def extract_clauses_e(sql):
    s=sql.lower()
    return {c for c in SQL_CLAUSES_E if re.search(r'\b'+c.replace(' ',r'\s+')+r'\b',s)}

def extract_tables_e(sql):
    return {m.group(1) for m in re.finditer(
        r'(?:from|join)\s+([a-zA-Z_][a-zA-Z0-9_]*)', sql.lower())}

def classify_error(pred, ref, tf1):
    pred = strip_cot(pred)
    if not is_valid_hive(pred):             return 'SYNTAX_ERROR'
    pc,rc = extract_clauses_e(pred), extract_clauses_e(ref)
    if rc-pc:                               return 'MISSING_CLAUSE'
    if pc-rc:                               return 'EXTRA_CLAUSE'
    if extract_tables_e(pred)!=extract_tables_e(ref): return 'WRONG_TABLE'
    if set(re.findall(r'[a-zA-Z_][a-zA-Z0-9_]*',pred.lower())) !=        set(re.findall(r'[a-zA-Z_][a-zA-Z0-9_]*',ref.lower())):
        return 'WRONG_COLUMN'
    if set(re.findall(r"'[^']*'|\b\d+\.?\d*\b",normalise_sql(pred))) !=        set(re.findall(r"'[^']*'|\b\d+\.?\d*\b",normalise_sql(ref))):
        return 'VALUE_ERROR'
    if pc!=rc:                              return 'WRONG_CLAUSE'
    if tf1>=0.7:                            return 'PARTIAL_MATCH'
    return 'COMPLETE_MISMATCH'

for pipe, preds in [('baseline',preds_baseline),('full',preds_full)]:
    err_mask = ~test_eval[f'em_{pipe}']
    edf = test_eval[err_mask].copy()
    edf['error_type'] = [classify_error(preds[i], test_refs[i],
                                         test_eval[f'tf1_{pipe}'].iloc[i])
                          for i in edf.index]
    test_eval.loc[err_mask, f'etype_{pipe}'] = edf['error_type'].values

full_err  = test_eval[~test_eval['em_full']]['etype_full'].value_counts()
base_err  = test_eval[~test_eval['em_baseline']]['etype_baseline'].value_counts()
all_types = sorted(set(full_err.index)|set(base_err.index))
comp = pd.DataFrame({'Error':all_types,
                      'Full pipeline':  [full_err.get(t,0) for t in all_types],
                      'Baseline':        [base_err.get(t,0) for t in all_types]})
print("Error comparison — Full pipeline vs Baseline:")
print(comp.to_string(index=False))

fig,axes = plt.subplots(1,2,figsize=(14,5))
axes[0].pie(full_err.values,labels=full_err.index,autopct='%1.1f%%',startangle=140)
axes[0].set_title('Full pipeline errors',fontweight='bold')
x=np.arange(len(comp)); w=0.38
axes[1].bar(x-w/2, comp['Full pipeline'], w, label='Full pipeline', color='steelblue',alpha=0.85)
axes[1].bar(x+w/2, comp['Baseline'],       w, label='Baseline',      color='coral',    alpha=0.85)
axes[1].set_xticks(x); axes[1].set_xticklabels(comp['Error'],rotation=30,ha='right')
axes[1].legend(); axes[1].set_title('Error counts — Full vs Baseline',fontweight='bold')
plt.suptitle('Error Taxonomy',fontsize=13,fontweight='bold')
plt.tight_layout()
plt.savefig('m5_error_taxonomy.png', bbox_inches='tight')
plt.show()
print("Saved → m5_error_taxonomy.png")


---
## 8  Clause-Level Accuracy

In [ ]:
TRACKED_C = ['select','from','where','group by','having','order by','limit','join','union']

def clause_prf(pred_col_preds):
    stats={}
    for clause in TRACKED_C:
        tp=fp=fn=tn=0
        for i,row in test_eval.iterrows():
            ph = clause in extract_clauses_e(strip_cot(pred_col_preds[i]))
            rh = clause in extract_clauses_e(test_refs[i])
            if ph and rh: tp+=1
            elif ph:      fp+=1
            elif rh:      fn+=1
            else:         tn+=1
        pr = tp/(tp+fp) if tp+fp>0 else 0
        rc = tp/(tp+fn) if tp+fn>0 else 0
        f1 = 2*pr*rc/(pr+rc) if pr+rc>0 else 0
        stats[clause]={'P':round(pr*100,1),'R':round(rc*100,1),
                        'F1':round(f1*100,1),'sup':tp+fn}
    return pd.DataFrame(stats).T.reset_index().rename(columns={'index':'Clause'})

full_cl = clause_prf(preds_full)
base_cl = clause_prf(preds_baseline)
full_cl = full_cl[full_cl['sup']>0].sort_values('F1',ascending=False)

merged = full_cl[['Clause','F1']].merge(
    base_cl[['Clause','F1']], on='Clause', suffixes=('_Full','_Base'))
merged['Delta'] = merged['F1_Full'] - merged['F1_Base']
print("Clause F1 — Full pipeline vs Baseline:")
print(merged.to_string(index=False))

fig,axes=plt.subplots(1,2,figsize=(14,max(4,len(full_cl)*0.55)))
for ax,data,title in zip(axes,[
    full_cl.set_index('Clause')[['P','R','F1']],
    base_cl.set_index('Clause')[['P','R','F1']].reindex(full_cl.set_index('Clause').index)],
    ['Full pipeline','Baseline']):
    sns.heatmap(data,annot=True,fmt='.1f',cmap='RdYlGn',
                vmin=0,vmax=100,linewidths=0.5,ax=ax,annot_kws={'size':9})
    ax.set_title(f'Clause Accuracy — {title}',fontweight='bold')
plt.tight_layout()
plt.savefig('m5_clause_heatmap.png',bbox_inches='tight')
plt.show()
print("Saved → m5_clause_heatmap.png")


---
## 9  Qualitative Inspection

In [ ]:
# What the full pipeline fixed vs baseline
fixed = test_eval[test_eval['em_full'] & ~test_eval['em_baseline']].head(5)
broke = test_eval[~test_eval['em_full'] &  test_eval['em_baseline']].head(5)

print(f"Full pipeline FIXED {len(test_eval[test_eval['em_full']&~test_eval['em_baseline']])} examples that Baseline got wrong")
print(f"Full pipeline BROKE {len(test_eval[~test_eval['em_full']& test_eval['em_baseline']])} examples that Baseline got right\n")

print("="*80+"\nFIXED BY FULL PIPELINE (FSP+CoT helped)\n"+"="*80)
for i,(_,row) in enumerate(fixed.iterrows(),1):
    idx = row.name
    print(f"[{i}] Input    : {row['sqlite'][:100]}")
    print(f"     Target   : {test_refs[idx][:100]}")
    print(f"     Full     : {strip_cot(preds_full[idx])[:100]}")
    print(f"     Baseline : {preds_baseline[idx][:100]}")
    print()

print("="*80+"\nBROKE BY FULL PIPELINE (FSP/CoT introduced error)\n"+"="*80)
for i,(_,row) in enumerate(broke.iterrows(),1):
    idx = row.name
    print(f"[{i}] Input    : {row['sqlite'][:100]}")
    print(f"     Target   : {test_refs[idx][:100]}")
    print(f"     Full     : {strip_cot(preds_full[idx])[:100]}")
    print(f"     Baseline : {preds_baseline[idx][:100]}")
    if f'etype_full' in test_eval.columns:
        print(f"     Error    : {test_eval.loc[row.name,'etype_full']}")
    print()


---
## 10  Attention Visualisation

In [ ]:
def plot_attention(model, src_text, tgt_text, layer=-1, head=0, max_len=48):
    model.eval()
    enc = tokenizer(src_text,return_tensors='pt',max_length=max_len,truncation=True).to(device)
    dec = tokenizer(tgt_text,return_tensors='pt',max_length=max_len,truncation=True).to(device)
    with torch.no_grad():
        out = model(input_ids=enc['input_ids'],attention_mask=enc['attention_mask'],
                    labels=dec['input_ids'],output_attentions=True)
    ca = out.cross_attentions[layer][0,head].cpu().numpy()
    src_toks=[t.replace('▁','') for t in tokenizer.convert_ids_to_tokens(enc['input_ids'][0])]
    tgt_toks=[t.replace('▁','') for t in tokenizer.convert_ids_to_tokens(dec['input_ids'][0])]
    fig,ax=plt.subplots(figsize=(min(len(src_toks)*0.55+2,18),min(len(tgt_toks)*0.45+2,12)))
    im=ax.imshow(ca[:len(tgt_toks),:len(src_toks)],cmap='YlOrRd',aspect='auto',vmin=0)
    ax.set_xticks(range(len(src_toks))); ax.set_xticklabels(src_toks,rotation=45,ha='right',fontsize=8)
    ax.set_yticks(range(len(tgt_toks))); ax.set_yticklabels(tgt_toks,fontsize=8)
    ax.set_title(f'Cross-attention layer={layer} head={head}\n{src_text[:60]}',fontsize=9,fontweight='bold')
    plt.colorbar(im,ax=ax,fraction=0.02)
    plt.tight_layout()
    plt.savefig('m5_attention_heatmap.png',bbox_inches='tight')
    plt.show()
    print("Saved → m5_attention_heatmap.png")

# Use shortest test query for clarity
idx = test_eval['sqlite'].apply(lambda x: len(x.split())).idxmin()
q   = test_df.loc[idx,'sqlite']
h   = test_df.loc[idx,'hive']
chunks = retrieve_and_rerank(q)
exs    = retrieve_fsp_examples(q,n=2)
src = (train_config['src_pfx']+q+' '+train_config['ctx_pfx']+' '+
       build_context_string(chunks)+' '+format_fsp_block(exs)+' '+
       train_config['cot_trigger'])
tgt = train_config['tgt_pfx']+h
print(f"Visualising attention:\n  src: {src[:80]}...")
plot_attention(best_model, src, tgt)


---
## 11  M4 Experiment Comparison

In [ ]:
if os.path.exists('milestone4_experiment_log.csv'):
    exp_df = pd.read_csv('milestone4_experiment_log.csv').dropna(subset=['val_exact_match'])

    # Colour by pipeline; LoRA experiments get a darker shade of their pipeline colour
    PIPE_COLORS = {
        'baseline' : '#4C72B0',
        'rag'      : '#DD8452',
        'fsp'      : '#55A868',
        'cot'      : '#C44E52',
        'full'     : '#9B59B6',
    }
    LORA_SUFFIX_COLOR = '#2ecc71'  # bright green border for LoRA rows

    bar_colors = []
    for _, row in exp_df.iterrows():
        pipe = str(row.get('pipeline','baseline')).lower()
        bar_colors.append(PIPE_COLORS.get(pipe, '#888780'))

    fig, axes = plt.subplots(1, 3, figsize=(17, max(6, len(exp_df)*0.35)))
    for ax, metric, label in zip(axes,
        ['val_exact_match','val_bleu','val_parse_valid'],
        ['Exact Match %','BLEU','Parse Validity %']):
        bars = ax.barh(exp_df['name'], exp_df[metric],
                       color=bar_colors, edgecolor='white', alpha=0.85)
        # Add green edge for LoRA experiments
        if 'use_lora' in exp_df.columns:
            for bar, is_lora in zip(bars, exp_df['use_lora'].fillna(False)):
                if is_lora:
                    bar.set_edgecolor(LORA_SUFFIX_COLOR)
                    bar.set_linewidth(1.5)
        ax.set_xlabel(label); ax.set_title(label, fontweight='bold')
        ax.axvline(exp_df[metric].max(), color='red', lw=1.5, linestyle='--')

    # Legend: pipelines + LoRA indicator
    import matplotlib.patches as mpatches
    handles = [mpatches.Patch(color=c, label=p) for p, c in PIPE_COLORS.items()]
    handles.append(mpatches.Patch(facecolor='white',
                                   edgecolor=LORA_SUFFIX_COLOR, linewidth=2,
                                   label='LoRA (green border)'))
    fig.legend(handles=handles, loc='upper right', title='Pipeline', fontsize=9)
    plt.suptitle('All M4 Experiments — Pipeline + LoRA Coloured',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('m5_experiment_comparison.png', bbox_inches='tight')
    plt.show()
    print("Saved → m5_experiment_comparison.png")

    # ── LoRA vs full fine-tune summary ────────────────────────────────────
    if 'use_lora' in exp_df.columns:
        lora_df = exp_df[exp_df['use_lora'].astype(str).isin(['True','1','true'])]
        full_df = exp_df[~exp_df['use_lora'].astype(str).isin(['True','1','true'])]
        print("\nLoRA experiments summary:")
        if len(lora_df):
            print(lora_df[['name','pipeline','val_exact_match','val_bleu',
                            'trainable_params']].to_string(index=False))
        print("\nFull fine-tune experiments summary:")
        if len(full_df):
            print(full_df[['name','pipeline','val_exact_match','val_bleu']].to_string(index=False))
else:
    print("milestone4_experiment_log.csv not found — run M4 experiments first.")


---
## 12  Save Evaluation Report

In [ ]:
save_cols = ['sqlite','hive']
# Add LoRA experiment predictions if they were generated
lora_pipe_names = ['lora_baseline','lora_rag','lora_full']

for pipe in ['baseline','rag','fsp','cot','full']:
    for col in [f'em_{pipe}',f'pv_{pipe}',f'tf1_{pipe}']:
        if col in test_eval.columns: save_cols.append(col)
if 'complexity'   in test_eval.columns: save_cols.append('complexity')
if 'is_synthetic' in test_eval.columns: save_cols.append('is_synthetic')
for pipe in ['baseline','full']:
    col=f'etype_{pipe}'
    if col in test_eval.columns: save_cols.append(col)

# Add predictions
for pipe, preds in [('baseline',preds_baseline),('rag',preds_rag),
                     ('fsp',preds_fsp),('cot',preds_cot),('full',preds_full)]:
    test_eval[f'pred_{pipe}'] = [strip_cot(p) for p in preds]
    save_cols.append(f'pred_{pipe}')

report = test_eval[[c for c in save_cols if c in test_eval.columns]]
report.to_csv('milestone5_eval_report.csv',index=False)
print(f"Saved → milestone5_eval_report.csv  ({len(report)} rows, {len(report.columns)} cols)")

import json
agg = {
    'best_model': CKPT,
    'pipeline_metrics': {k:{m:v for m,v in pm.items() if not m.startswith('_')}
                          for k,pm in pipeline_metrics.items()},
    'retrieval_metrics': ret_metrics,
    'error_distribution_full':     full_err.to_dict(),
    'error_distribution_baseline': base_err.to_dict(),
}
with open('milestone5_metrics.json','w') as f: json.dump(agg,f,indent=2)
print("Saved → milestone5_metrics.json")


---
## 13  Limitations & Possible Improvements

### 13.1  Current limitations

| # | Area | Limitation | Impact |
|---|---|---|---|
| 1 | **FSP** | 512-token budget forces truncation when query + context + examples are all long | Complex queries get fewer few-shot examples |
| 2 | **FSP** | Rule-based heuristic relevance labels for pool selection; no ground-truth relevance judgements | May retrieve structurally dissimilar examples |
| 3 | **CoT** | Auto-annotator uses only keyword rules; misses semantic nuances (e.g. implicit type casts) | ~15-20% of CoT traces may be incomplete |
| 4 | **CoT** | Only 30% of training data annotated; rest trains on plain targets without reasoning | CoT behaviour inconsistent across query types |
| 5 | **RAG** | FAISS index is static; schema changes or new HiveQL functions require full re-indexing | Deployment drift over time |
| 6 | **RAG reranker** | `ms-marco-MiniLM` trained on web search, not SQL documentation retrieval | Cross-domain mismatch limits reranking gains |
| 7 | **Model** | T5-base (220M params) may not have enough capacity for complex multi-join queries with long CoT | CoT benefit capped by model capacity |
| 8 | **Evaluation** | Exact Match penalises semantically equivalent but differently formatted HiveQL | Underestimates true translation quality |

---

### 13.2  Possible improvements

**Few-shot prompting**
- Fine-tune a retriever on `(query, good_example)` pairs to improve example selection
- Experiment with 1-shot vs 3-shot; larger models can absorb 3 examples without truncation
- Use complexity-stratified example pools so simple queries only get simple examples

**Chain-of-thought**
- Manually annotate 100-200 high-quality CoT traces for complex queries, then distil to the rest
- Train a separate CoT-generator LLM (e.g. GPT-3.5) to annotate the full training set
- Use execution-guided CoT: include HiveQL parse result in the trace to signal correctness

**RAG**
- Fine-tune bi-encoder on SQLite query ↔ HiveQL doc relevance pairs (domain adaptation)
- Switch to `BAAI/bge-base-en-v1.5` for stronger embeddings
- Add incremental FAISS updates via `IndexIDMap` to support live schema changes

**Architecture**
- CodeT5+ (pre-trained on GitHub code) has much better SQL token priors than T5
- T5-large or T5-3B for higher capacity on complex reasoning + translation

**Inference**
- Constrained decoding with a HiveQL grammar mask (prevents syntactically invalid tokens)
- Self-consistency: generate N candidates, pick by parse validity + self-BLEU
- Post-processing rules: normalise quoting, capitalise keywords
